# Phase 3 — Baseline B0…B3 cho fusion (nhãn `ketqua`)

**Chạy lần lượt từng cell từ trên xuống.** Tổng thời gian khoảng 10–15 phút.

| Baseline | Đầu vào | Trả lời câu hỏi |
|---|---|---|
| **B0** | Chỉ lâm sàng (bảng 139 cột + điểm nhánh ts) | Hồ sơ lâm sàng làm được tới đâu |
| **B1** | Chỉ ảnh, riêng 1.835 người có ảnh | Ảnh làm được tới đâu |
| **B2** | Lâm sàng + ảnh, **không** dùng đồ thị | Hợp nhất có hơn từng nhánh không |
| **B3** | B2 + đặc trưng ảnh trung bình của hàng xóm | **Người không có ảnh có học được từ hàng xóm không** |

B3 chính là giả thuyết trung tâm của đồ án, nhưng gộp hàng xóm một lần sẵn chứ chưa
phải GNN. Phase 4 (GAT) sẽ phải vượt B3 thì GNN mới có lý do tồn tại.

> Trước khi chạy: đảm bảo đã **commit và push** nhánh `fusion_gnn` ở máy local.
> Notebook này tải code và dữ liệu từ GitHub.

## 1. Bật GPU và kiểm tra

Menu **Runtime → Change runtime type → T4 GPU → Save**, rồi chạy cell dưới.

In [ ]:
import torch
print("Torch:", torch.__version__)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG THẤY GPU — vào Runtime → Change runtime type → T4 GPU")

## 2. Tải mã nguồn và dữ liệu

Nếu repo ở chế độ **private**, cell này sẽ hỏi username + Personal Access Token.

In [ ]:
import os, shutil
if os.path.exists("/content/repo"):
    shutil.rmtree("/content/repo")
!git clone -q -b fusion_gnn https://github.com/Do-Trung-Quan/DO-AN-TOT-NGHIEP-E22-T02.git /content/repo
os.chdir("/content/repo")
print("Thư mục làm việc:", os.getcwd())

# Kiểm tra đủ file đầu vào trước khi chạy — thiếu file nào sẽ báo ngay ở đây
need = [
    "fusion/output/fusion_dataset_ketqua_control.npz",
    "fusion/output/fusion_dataset_ketqua_frozen.npz",
    "fusion/output/graph_ketqua_k10_bridge.npz",
    "fusion/output/tabular_ketqua.npz",
    "output/ketqua/ts_predictions.parquet",
]
missing = [f for f in need if not os.path.exists(f)]
print("THIẾU:", missing) if missing else print("✅ Đủ toàn bộ file đầu vào")

## 3. Chạy baseline — nguồn ảnh `control`

Đây là nguồn ảnh chính. Khoảng 5–8 phút trên T4.

`--seeds 3` nghĩa là mỗi fold huấn luyện 3 lần với 3 seed rồi lấy trung bình dự đoán,
để kết quả không phụ thuộc may rủi của một lần khởi tạo.

In [ ]:
!python fusion/baselines.py --source control --seeds 3 --epochs 60

## 4. Xem lại kết quả dưới dạng bảng

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("fusion/output/phase3_baselines.md", encoding="utf-8").read()))

## 5. (Tuỳ chọn) Chạy thêm nguồn ảnh `frozen`

Dùng để đối chiếu: `frozen` là BioViL-T gốc, chưa học nhãn nào. Nếu `control` không hơn
`frozen` thì việc fine-tune trên SetA không mang lại gì.

⚠️ Cell này **ghi đè** file kết quả của `control`, nên hãy tải file ở bước 6 trước.

In [ ]:
!python fusion/baselines.py --source frozen --seeds 3 --epochs 60
!cp fusion/output/phase3_baselines.md   fusion/output/phase3_baselines_frozen.md
!cp fusion/output/phase3_baselines.json fusion/output/phase3_baselines_frozen.json
print("Đã lưu bản frozen ra file riêng")

## 6. Tải kết quả về máy

Tải xong thì copy vào đúng thư mục `fusion/output/` ở repo local rồi commit.

**Không `git push` từ Colab** — dán token lên Colab là rủi ro lộ khoá.

In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob("fusion/output/phase3_*")):
    print("Đang tải:", f)
    files.download(f)